<a href="https://colab.research.google.com/github/rohanmyers/queue-viability/blob/main/notebooks/01_data_inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geopandas shap xgboost pyarrow openpyxl

import pandas as pd
import geopandas as gpd
print(f"pandas {pd.__version__}, geopandas {gpd.__version__}")

Mounted at /content/drive
pandas 2.2.2, geopandas 1.1.3


In [2]:
import os
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path("/content/drive/MyDrive/queue-viability")
RAW = DRIVE_ROOT / "data" / "raw"
INTERIM = DRIVE_ROOT / "data" / "interim"
PROCESSED = DRIVE_ROOT / "data" / "processed"
FIGURES = DRIVE_ROOT / "reports" / "figures"

for p in [RAW, INTERIM, PROCESSED, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

# List what's actually in raw/
print("Files in data/raw/:")
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:>8.2f} MB")

Files in data/raw/:
  eia860_2024.zip                              22.10 MB
  eia861_2024.zip                               4.57 MB
  hifld_substations.geojson                     0.07 MB
  hifld_transmission_lines.geojson            159.02 MB
  hifld_transmission_lines.zip                 39.58 MB
  inventory.csv                                 0.00 MB
  lbnl_queued_up_2025.xlsx                     14.07 MB


In [3]:
inventory = [
    {
        "dataset": "LBNL Queued Up",
        "filename": "lbnl_queued_up_2025.xlsx",
        "source_url": "https://emp.lbl.gov/queues",
        "downloaded_on": "2026-04-24",
        "purpose": "Historical and active interconnection queue — primary training data"
    },
    {
        "dataset": "HIFLD Electric Substations",
        "filename": "hifld_substations.geojson",
        "source_url": "https://catalog.data.gov/dataset/electric-substations",
        "downloaded_on": "2026-04-24",
        "purpose": "Feature: distance to nearest 230+ kV substation"
    },
    {
        "dataset": "HIFLD Transmission Lines",
        "filename": "hifld_transmission_lines.geojson",
        "source_url": "https://www.datalumos.org/datalumos/project/240591/version/V1/view",
        "downloaded_on": "2026-04-24",
        "purpose": "Feature: transmission capacity proxies (Day 3+)"
    },
    {
        "dataset": "EIA Form 860 (2023)",
        "filename": "eia860_2024.zip",
        "source_url": "https://www.eia.gov/electricity/data/eia860/",
        "downloaded_on": "2026-04-24",
        "purpose": "Label verification: which queued projects are now operational"
    },
    {
        "dataset": "EIA Form 861 (2023)",
        "filename": "eia861_2024.zip",
        "source_url": "https://www.eia.gov/electricity/data/eia861/",
        "downloaded_on": "2026-04-24",
        "purpose": "Feature: utility-level retail electricity price"
    },
    {
        "dataset": "datacenter.fyi",
        "filename": "TBD",
        "source_url": "https://www.interconnection.fyi/data-center",
        "downloaded_on": "pending",
        "purpose": "Label confirmation for data-center-specific requests — address Day 3"
    },
]

inv_df = pd.DataFrame(inventory)

# Add actual file size and existence check
inv_df["file_exists"] = inv_df["filename"].apply(
    lambda f: (RAW / f).exists() if f != "TBD" else False
)
inv_df["size_mb"] = inv_df["filename"].apply(
    lambda f: round((RAW / f).stat().st_size / 1e6, 2) if f != "TBD" and (RAW / f).exists() else None
)

inv_df.to_csv(RAW / "inventory.csv", index=False)
inv_df

,dataset,filename,source_url,downloaded_on,purpose,file_exists,size_mb
0,LBNL Queued Up,lbnl_queued_up_2025.xlsx,https://emp.lbl.gov/queues,2026-04-24,Historical and active interconnection queue — ...,True,14.07
1,HIFLD Electric Substations,hifld_substations.geojson,https://catalog.data.gov/dataset/electric-subs...,2026-04-24,Feature: distance to nearest 230+ kV substation,True,0.07
2,HIFLD Transmission Lines,hifld_transmission_lines.geojson,https://www.datalumos.org/datalumos/project/24...,2026-04-24,Feature: transmission capacity proxies (Day 3+),True,159.02
3,EIA Form 860 (2023),eia860_2024.zip,https://www.eia.gov/electricity/data/eia860/,2026-04-24,Label verification: which queued projects are ...,True,22.10
4,EIA Form 861 (2023),eia861_2024.zip,https://www.eia.gov/electricity/data/eia861/,2026-04-24,Feature: utility-level retail electricity price,True,4.57
5,datacenter.fyi,TBD,https://www.interconnection.fyi/data-center,pending,Label confirmation for data-center-specific re...,False,NaN


In [4]:
xl = pd.ExcelFile(RAW / "lbnl_queued_up_2025.xlsx")
print("Sheets in LBNL file:")
for s in xl.sheet_names:
    df = pd.read_excel(RAW / "lbnl_queued_up_2025.xlsx", sheet_name=s, nrows=0)
    nrows = len(pd.read_excel(RAW / "lbnl_queued_up_2025.xlsx", sheet_name=s))
    print(f"  {s}: {nrows} rows, {len(df.columns)} cols")

Sheets in LBNL file:
  Introduction: 38 rows, 0 cols
  Contents: 39 rows, 3 cols
  00. Background + Methods: 20 rows, 1 cols
  01. Balancing Areas: 19 rows, 5 cols
  02. Data Sample by Region: 18 rows, 1 cols
  03. Complete Queue Data: 36442 rows, 1 cols
  04. Data Codebook: 36 rows, 1 cols
  05. Annual Requests: 47 rows, 1 cols
  06. Active Capacity by Year: 44 rows, 1 cols
  07. Active Capacity by Type: 200 rows, 1 cols
  08. Active Cap. Region+Type: 794 rows, 1 cols
  09. Queues vs. Installed: 49 rows, 1 cols
  10. Active Cap. Maps: 273 rows, 1 cols
  11. Ix. Request Size Trends: 79 rows, 1 cols
  12. Other Gen. + Storage: 41 rows, 1 cols
  13. Hybrid Capacity: 44 rows, 1 cols
  14. ERIS + NRIS Capacity: 75 rows, 1 cols
  15. Cap. by Prop. Online Year: 34 rows, 1 cols
  16. Cap. by Ix. Phase: 31 rows, 1 cols
  17. Cap. with IA: 35 rows, 1 cols
  18. IA Throughput by Region: 74 rows, 1 cols
  19. Operational Volume Trend: 49 rows, 1 cols
  20. Withdrawn Volume Trend: 49 rows, 1 cols


In [5]:
subs = gpd.read_file(RAW / "hifld_substations.geojson")
print(f"Substations: {len(subs)}")
print(f"CRS: {subs.crs}")
print(f"Columns: {subs.columns.tolist()[:15]}")
subs.head(3)

Substations: 19
CRS: EPSG:4326
Columns: ['OBJECTID', 'ELP', 'ELP_ID', 'ELP_CODE', 'ELP_NAME', 'DXF_LAYER', 'DESC_', 'SHAPEAREA', 'SHAPELEN', 'geometry']


,OBJECTID,ELP,ELP_ID,ELP_CODE,ELP_NAME,DXF_LAYER,DESC_,SHAPEAREA,SHAPELEN,geometry
0,1,2,3,7110,,SUBSTATION,Substation,0,0,"POLYGON ((-76.95657 38.90068, -76.95656 38.900..."
1,2,3,4,7110,,SUBSTATION,Substation,0,0,"POLYGON ((-76.9562 38.90016, -76.9563 38.90016..."
2,3,11,12,7110,,SUBSTATION,Substation,0,0,"POLYGON ((-76.95256 38.89935, -76.95265 38.899..."


In [6]:
import zipfile

for zip_name in ["eia860_2024.zip", "eia861_2024.zip"]:
    print(f"\n=== {zip_name} ===")
    with zipfile.ZipFile(RAW / zip_name) as z:
        for n in z.namelist()[:8]:
            print(f"  {n}")


=== eia860_2024.zip ===
  1___Utility_Y2024.xlsx
  2___Plant_Y2024.xlsx
  3_1_Generator_Y2024.xlsx
  3_2_Wind_Y2024.xlsx
  3_3_Solar_Y2024.xlsx
  3_4_Energy_Storage_Y2024.xlsx
  3_5_Multifuel_Y2024.xlsx
  4___Owner_Y2024.xlsx

=== eia861_2024.zip ===
  Service_Territory_2024.xlsx
  Short_Form_2024.xlsx
  Utility_Data_2024.xlsx
  2024 EIA-861 Instructions.pdf
  2024 EIA-861 Form.xlsx
  Advanced_Meters_2024.xlsx
  Balancing_Authority_2024.xlsx
  Delivery_Companies_2024.xlsx
